In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/superstore_clean.csv")

# SQL is easier with simple column names (no spaces or hyphens)
df.columns = df.columns.str.lower().str.replace("-", "_").str.replace(" ", "_")

conn = sqlite3.connect("../data/superstore.db")
df.to_sql("superstore", conn, if_exists="replace", index=False)

print(df.columns.tolist())

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'year', 'month', 'month_name', 'shipping_days']


In [2]:
def q(sql):
    return pd.read_sql(sql, conn)

In [8]:
#Sanity check
q("""
SELECT COUNT(*) AS row_count,
       ROUND(SUM(sales), 2) AS total_sales,
       COUNT(DISTINCT order_id) AS orders,
       COUNT(DISTINCT customer_id) AS customers
FROM superstore;
""")

,row_count,total_sales,orders,customers
0,9800,2261536.78,4922,793


In [9]:
#Sales by region
q("""
SELECT region, ROUND(SUM(sales), 2) AS total_sales
FROM superstore
GROUP BY region
ORDER BY total_sales DESC;
""")

,region,total_sales
0,West,710219.68
1,East,669518.73
2,Central,492646.91
3,South,389151.46


In [ ]:
#Yearly sales with growth
q("""
WITH yearly AS (
    SELECT year, SUM(sales) AS total_sales
    FROM superstore
    GROUP BY year
)
SELECT year,
       ROUND(total_sales, 2) AS total_sales,
       ROUND((total_sales - LAG(total_sales) OVER (ORDER BY year))
             * 100.0 / LAG(total_sales) OVER (ORDER BY year), 1) AS yoy_growth_pct
FROM yearly
ORDER BY year;
""")

,year,total_sales,yoy_growth_pct
0,2015,479856.21,NaN
1,2016,459436.01,-4.3
2,2017,600192.55,30.6
3,2018,722052.02,20.3


In [ ]:
#average sales per month
q("""
SELECT month,
       ROUND(SUM(sales) / COUNT(DISTINCT year), 0) AS avg_monthly_sales
FROM superstore
GROUP BY month
ORDER BY avg_monthly_sales DESC;
""")

,month,avg_monthly_sales
0,11,87540.0
1,12,80370.0
2,9,75026.0
3,10,49874.0
4,3,49393.0
5,8,39329.0
6,5,38522.0
7,6,36459.0
8,7,36384.0
9,4,34071.0


In [ ]:
#Category and sub-category share of sales
q("""
SELECT category,
       sub_category,
       ROUND(SUM(sales), 0) AS total_sales,
       ROUND(SUM(sales) * 100.0 / SUM(SUM(sales)) OVER (), 1) AS pct_of_total
FROM superstore
GROUP BY category, sub_category
ORDER BY total_sales DESC;
""")

,category,sub_category,total_sales,pct_of_total
0,Technology,Phones,327782.0,14.5
1,Furniture,Chairs,322823.0,14.3
2,Office Supplies,Storage,219343.0,9.7
3,Furniture,Tables,202811.0,9.0
4,Office Supplies,Binders,200029.0,8.8
5,Technology,Machines,189239.0,8.4
6,Technology,Accessories,164187.0,7.3
7,Technology,Copiers,146248.0,6.5
8,Furniture,Bookcases,113813.0,5.0
9,Office Supplies,Appliances,104618.0,4.6


In [10]:
#Top 10 customers
q("""
SELECT customer_name,
       ROUND(SUM(sales), 2) AS total_sales,
       COUNT(DISTINCT order_id) AS orders
FROM superstore
GROUP BY customer_id, customer_name
ORDER BY total_sales DESC
LIMIT 10;
""")

,customer_name,total_sales,orders
0,Sean Miller,25043.05,5
1,Tamara Chand,19052.22,5
2,Raymond Buch,15117.34,6
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,10
5,Ken Lonsdale,14175.23,12
6,Sanjit Chand,14142.33,9
7,Hunter Lopez,12873.30,6
8,Sanjit Engle,12209.44,11
9,Christopher Conant,12129.07,5


In [11]:
#Shipping time by ship mode
q("""
SELECT ship_mode,
       ROUND(AVG(shipping_days), 2) AS avg_days,
       COUNT(*) AS lines
FROM superstore
GROUP BY ship_mode
ORDER BY avg_days;
""")

,ship_mode,avg_days,lines
0,Same Day,0.04,538
1,First Class,2.18,1501
2,Second Class,3.25,1902
3,Standard Class,5.01,5859


## Summary
The SQL results match the pandas analysis: [one line, e.g. West leads regional sales, Nov/Dec/Sep are the peak months, sales grew about 50% from 2015 to 2018]. This confirms the findings using two different tools.